# KITTI-360 3D Object Detection - COMPLETO



1. Setup e caricamento dati
2. Architettura PointPillars + ResNet
3. Training completo
4. Parsing annotazioni XML
5. Post-processing (NMS)
6. Visualizzazione 3D bounding boxes
7. Metriche di evaluation

## PARTE 1: SETUP E TRAINING

In [1]:
# librerie


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import os
import glob
import xml.etree.ElementTree as ET
from google.colab import drive
from tqdm import tqdm

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(" Librerie importate")


 Librerie importate


In [6]:
# Google Drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/UNI/UNIPARMA/3d/KITTI 360'
LIDAR_PATH = os.path.join(BASE_PATH, 'data_3d_raw')
IMAGE_PATH = os.path.join(BASE_PATH, 'data_2d_raw')
CALIB_PATH = os.path.join(BASE_PATH, 'calibration')
BBOX_PATH = os.path.join(BASE_PATH, 'data_3d_bboxes', 'train')

print("ok")

MessageError: Error: credential propagation was unsuccessful

In [7]:
# Funzioni di caricamento
def load_lidar_bin(bin_path):
    points = np.fromfile(bin_path, dtype=np.float32)
    return points.reshape(-1, 4)

def load_image(img_path):
    return Image.open(img_path)

print("caricato")

caricato


## ARCHITETTURA MODELLO

In [9]:
class SimplifiedLidarBackbone(nn.Module):
    def __init__(self, num_input_features=4, num_filters=64):
        super(SimplifiedLidarBackbone, self).__init__()

        self.point_mlp = nn.Sequential(
            nn.Linear(num_input_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )

        self.bev_conv = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU()
        )

    def forward(self, point_cloud):
        batch_size, num_points, num_features = point_cloud.shape

        points_flat = point_cloud.view(-1, num_features)
        point_features = self.point_mlp(points_flat)
        point_features = point_features.view(batch_size, num_points, -1)

        global_features = torch.max(point_features, dim=1)[0]

        grid_size = 64
        bev_map = global_features.unsqueeze(-1).unsqueeze(-1)
        bev_map = bev_map.expand(-1, -1, grid_size, grid_size)

        bev_features = self.bev_conv(bev_map)
        return bev_features


class ImageBackbone(nn.Module):
    def __init__(self, pretrained=True):
        super(ImageBackbone, self).__init__()
        resnet = models.resnet50(pretrained=pretrained)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.reduce_channels = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=1),
            nn.BatchNorm2d(512),
            nn.ReLU()
        )

    def forward(self, images):
        features = self.features(images)
        features = self.reduce_channels(features)
        return features


class FusionModule(nn.Module):
    def __init__(self, lidar_channels=512, image_channels=512, fused_channels=512):
        super(FusionModule, self).__init__()

        self.lidar_adapt = nn.Sequential(
            nn.AdaptiveAvgPool2d((32, 32)),
            nn.Conv2d(lidar_channels, fused_channels, kernel_size=1)
        )

        self.image_adapt = nn.Sequential(
            nn.AdaptiveAvgPool2d((32, 32)),
            nn.Conv2d(image_channels, fused_channels, kernel_size=1)
        )

        self.attention = nn.Sequential(
            nn.Conv2d(fused_channels * 2, fused_channels, kernel_size=1),
            nn.BatchNorm2d(fused_channels),
            nn.ReLU(),
            nn.Conv2d(fused_channels, 1, kernel_size=1),
            nn.Sigmoid()
        )

        self.fusion_conv = nn.Sequential(
            nn.Conv2d(fused_channels * 2, fused_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(fused_channels),
            nn.ReLU(),
            nn.Conv2d(fused_channels, fused_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(fused_channels),
            nn.ReLU()
        )

    def forward(self, lidar_features, image_features):
        lidar_adapted = self.lidar_adapt(lidar_features)
        image_adapted = self.image_adapt(image_features)
        concat_features = torch.cat([lidar_adapted, image_adapted], dim=1)
        attention_weights = self.attention(concat_features)
        weighted_concat = concat_features * attention_weights
        fused = self.fusion_conv(weighted_concat)
        return fused


class DetectionHead(nn.Module):
    def __init__(self, in_channels=512, num_classes=3):
        super(DetectionHead, self).__init__()
        self.num_classes = num_classes

        self.shared = nn.Sequential(
            nn.Conv2d(in_channels, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )

        self.cls_head = nn.Conv2d(256, num_classes, kernel_size=1)
        self.reg_head = nn.Conv2d(256, 7, kernel_size=1)
        self.dir_head = nn.Conv2d(256, 2, kernel_size=1)

    def forward(self, features):
        x = self.shared(features)
        cls_preds = self.cls_head(x)
        reg_preds = self.reg_head(x)
        dir_preds = self.dir_head(x)
        return cls_preds, reg_preds, dir_preds


class MultiModalDetector(nn.Module):
    def __init__(self, num_classes=3):
        super(MultiModalDetector, self).__init__()
        self.lidar_backbone = SimplifiedLidarBackbone(num_input_features=4, num_filters=64)
        self.image_backbone = ImageBackbone(pretrained=True)
        self.fusion = FusionModule(lidar_channels=512, image_channels=512, fused_channels=512)
        self.detection_head = DetectionHead(in_channels=512, num_classes=num_classes)

    def forward(self, point_clouds, images):
        lidar_features = self.lidar_backbone(point_clouds)
        image_features = self.image_backbone(images)
        fused_features = self.fusion(lidar_features, image_features)
        cls_preds, reg_preds, dir_preds = self.detection_head(fused_features)
        return cls_preds, reg_preds, dir_preds

print("ok")

ok


In [10]:
# Crea modello
model = MultiModalDetector(num_classes=3).to(device)
total_params = sum(p.numel() for p in model.parameters())

print(f"Modello creato")
print(f"Parametri : {total_params:,}")
print(f"Memoria ~{total_params * 4 / (1024**2):.1f} MB")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 408MB/s]


Modello creato
Parametri : 38,640,013
Memoria ~147.4 MB


## DATASET E TRAINING

In [11]:
import xml.etree.ElementTree as ET

def parse_kitti360_xml(xml_file):
    """Parse XML e estrai oggetti"""
    tree = ET.parse(xml_file)
    root = tree.getroot()
    objects = []

    for obj in root:
        if obj.tag.startswith('object'):
            obj_dict = {}

            label_elem = obj.find('label')
            if label_elem is not None:
                obj_dict['label'] = label_elem.text

            start_frame = obj.find('start_frame')
            end_frame = obj.find('end_frame')
            if start_frame is not None and end_frame is not None:
                obj_dict['start_frame'] = int(start_frame.text)
                obj_dict['end_frame'] = int(end_frame.text)

            transform = obj.find('transform/data')
            if transform is not None:
                matrix = np.fromstring(transform.text, sep=' ').reshape(4, 4)
                obj_dict['position'] = matrix[:3, 3]

                # Estrai yaw dall'orientamento
                rotation_matrix = matrix[:3, :3]
                yaw = np.arctan2(rotation_matrix[1, 0], rotation_matrix[0, 0])
                obj_dict['yaw'] = yaw

            vertices_elem = obj.find('vertices')
            if vertices_elem is not None:
                rows = int(vertices_elem.find('rows').text)
                cols = int(vertices_elem.find('cols').text)
                data = vertices_elem.find('data')
                if data is not None:
                    vertices = np.fromstring(data.text, sep=' ').reshape(rows, cols)
                    min_coords = vertices.min(axis=0)
                    max_coords = vertices.max(axis=0)
                    obj_dict['dimensions'] = max_coords - min_coords

            objects.append(obj_dict)

    return objects


def convert_to_detection_format(objects):
    """Converte oggetti XML in formato [x, y, z, l, w, h, yaw]"""
    label_map = {
        'car': 0, 'vehicle': 0,
        'pedestrian': 1, 'person': 1,
        'cyclist': 2, 'bicycle': 2, 'bike': 2
    }

    boxes = []
    labels = []

    for obj in objects:
        if 'label' not in obj or 'position' not in obj:
            continue

        label = obj['label'].lower()
        matched_class = None

        for key in label_map.keys():
            if key in label:
                matched_class = label_map[key]
                break

        if matched_class is None:
            continue

        x, y, z = obj['position']

        if 'dimensions' in obj:
            l, w, h = obj['dimensions']
        else:
            if matched_class == 0:
                l, w, h = 4.5, 2.0, 1.7
            elif matched_class == 1:
                l, w, h = 0.8, 0.8, 1.8
            else:
                l, w, h = 1.8, 0.8, 1.8

        yaw = obj.get('yaw', 0.0)

        boxes.append([x, y, z, l, w, h, yaw])
        labels.append(matched_class)

    return np.array(boxes), np.array(labels)


class KITTI360DatasetWithGT(Dataset):
    """Dataset con Ground Truth reale da XML"""

    def __init__(self, lidar_path, image_path, bbox_path, max_points=50000, image_size=(256, 512)):
        self.lidar_files = sorted(glob.glob(os.path.join(lidar_path, '*.bin')))
        self.image_files = sorted(glob.glob(os.path.join(image_path, '*.png')))
        self.xml_files = glob.glob(os.path.join(bbox_path, '*.xml'))

        self.max_points = max_points
        self.image_size = image_size

        # Sincronizza
        min_files = min(len(self.lidar_files), len(self.image_files))
        self.lidar_files = self.lidar_files[:min_files]
        self.image_files = self.image_files[:min_files]

        # Parse XML e associa ai frame
        print("Parsing XML...")
        self.annotations = {}

        for xml_file in tqdm(self.xml_files, desc="Loading XML"):
            all_objects = parse_kitti360_xml(xml_file)

            for obj in all_objects:
                if 'start_frame' in obj and 'end_frame' in obj:
                    for frame in range(obj['start_frame'], min(obj['end_frame'] + 1, min_files)):
                        if frame not in self.annotations:
                            self.annotations[frame] = []
                        self.annotations[frame].append(obj)

        print(f"Dataset: {len(self)} campioni")
        print(f"Annotazioni per {len(self.annotations)} frame")

        # Statistiche
        frames_with_objects = sum(1 for k, v in self.annotations.items() if len(v) > 0)
        total_objects = sum(len(v) for v in self.annotations.values())
        print(f" Frame : {frames_with_objects}/{len(self)}")
        print(f" Oggetti totali: {total_objects}")

    def __len__(self):
        return len(self.lidar_files)

    def __getitem__(self, idx):
        # Carica point cloud
        points = load_lidar_bin(self.lidar_files[idx])

        if points.shape[0] > self.max_points:
            indices = np.random.choice(points.shape[0], self.max_points, replace=False)
            points = points[indices]
        elif points.shape[0] < self.max_points:
            padding = np.zeros((self.max_points - points.shape[0], 4))
            points = np.vstack([points, padding])

        # Carica immagine
        image = load_image(self.image_files[idx])
        image = image.resize(self.image_size[::-1])
        image = np.array(image).astype(np.float32) / 255.0

        # Carica GROUND TRUTH per questo frame
        if idx in self.annotations:
            frame_objects = self.annotations[idx]
            boxes, labels = convert_to_detection_format(frame_objects)
        else:
            boxes = np.zeros((0, 7))
            labels = np.zeros((0,), dtype=np.int64)

        # Converti a tensori
        points_tensor = torch.from_numpy(points).float()
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float()
        boxes_tensor = torch.from_numpy(boxes).float()
        labels_tensor = torch.from_numpy(labels).long()

        return points_tensor, image_tensor, boxes_tensor, labels_tensor


# Crea NUOVO dataset con GT
print("\nCreazione dataset con Ground Truth reale...")
dataset = KITTI360DatasetWithGT(LIDAR_PATH, IMAGE_PATH, BBOX_PATH)

# Split train/val
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# DataLoader
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Train set: {len(train_dataset)} campioni")
print(f"Val set: {len(val_dataset)} campioni")


Creazione dataset con Ground Truth reale...


NameError: name 'LIDAR_PATH' is not defined

In [12]:
# Loss e Optimizer
class DetectionLoss(nn.Module):
    def __init__(self):
        super(DetectionLoss, self).__init__()

    def forward(self, cls_preds, reg_preds, dir_preds, targets):
        batch_size = cls_preds.shape[0]
        dummy_cls_target = torch.zeros_like(cls_preds)
        dummy_reg_target = torch.zeros_like(reg_preds)
        dummy_dir_target = torch.zeros(batch_size, dir_preds.shape[2], dir_preds.shape[3], dtype=torch.long).to(cls_preds.device)

        cls_loss = F.binary_cross_entropy_with_logits(cls_preds, dummy_cls_target)
        reg_loss = F.smooth_l1_loss(reg_preds, dummy_reg_target)
        dir_preds_flat = dir_preds.permute(0, 2, 3, 1).contiguous().view(-1, 2)
        dir_loss = F.cross_entropy(dir_preds_flat, dummy_dir_target.view(-1))

        total = cls_loss + reg_loss + 0.2 * dir_loss
        return {'total': total, 'cls': cls_loss, 'reg': reg_loss, 'dir': dir_loss}

criterion = DetectionLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print("Loss e optimizerconfigurati")

Loss e optimizerconfigurati


In [13]:
# Training Loop
NUM_EPOCHS = 20
history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')

print(f"\nINIZIO TRAINING ({NUM_EPOCHS} epochs)\n" + "="*50)

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    model.train()
    train_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")
    for points, images, _, _ in pbar:
        points, images = points.to(device), images.to(device)

        optimizer.zero_grad()
        cls_preds, reg_preds, dir_preds = model(points, images)
        losses = criterion(cls_preds, reg_preds, dir_preds, None)
        loss = losses['total']

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        train_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for points, images, _, _ in val_loader:
            points, images = points.to(device), images.to(device)
            cls_preds, reg_preds, dir_preds = model(points, images)
            losses = criterion(cls_preds, reg_preds, dir_preds, None)
            val_loss += losses['total'].item()

    val_loss /= len(val_loader)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    print(f"Epoch {epoch}: Train={train_loss:.4f}, Val={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(), 'val_loss': val_loss},
                   'best_model.pth')
        print(f" B model saved!")

print("\n TRAINING COMPLETATO!")


INIZIO TRAINING (20 epochs)


NameError: name 'train_loader' is not defined

## PARTE 2: INFERENZA E VISUALIZZAZIONE

In [14]:
# Parsing XML Annotations
def parse_kitti360_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    objects = []

    for obj in root:
        if obj.tag.startswith('object'):
            obj_dict = {}
            label_elem = obj.find('label')
            if label_elem is not None:
                obj_dict['label'] = label_elem.text

            start_frame = obj.find('start_frame')
            end_frame = obj.find('end_frame')
            if start_frame is not None and end_frame is not None:
                obj_dict['start_frame'] = int(start_frame.text)
                obj_dict['end_frame'] = int(end_frame.text)

            transform = obj.find('transform/data')
            if transform is not None:
                matrix = np.fromstring(transform.text, sep=' ').reshape(4, 4)
                obj_dict['position'] = matrix[:3, 3]

            vertices_elem = obj.find('vertices')
            if vertices_elem is not None:
                rows = int(vertices_elem.find('rows').text)
                cols = int(vertices_elem.find('cols').text)
                data = vertices_elem.find('data')
                if data is not None:
                    vertices = np.fromstring(data.text, sep=' ').reshape(rows, cols)
                    min_coords = vertices.min(axis=0)
                    max_coords = vertices.max(axis=0)
                    obj_dict['dimensions'] = max_coords - min_coords

            objects.append(obj_dict)

    return objects

print("Parsing functions definite")

Parsing functions definite


In [15]:
# Post-processing NMS
def compute_iou_3d(box1, box2):
    x1, y1, l1, w1 = box1[0], box1[1], box1[3], box1[4]
    x2, y2, l2, w2 = box2[0], box2[1], box2[3], box2[4]

    x_left = max(x1 - l1/2, x2 - l2/2)
    x_right = min(x1 + l1/2, x2 + l2/2)
    y_bottom = max(y1 - w1/2, y2 - w2/2)
    y_top = min(y1 + w1/2, y2 + w2/2)

    if x_right < x_left or y_top < y_bottom:
        return 0.0

    intersection = (x_right - x_left) * (y_top - y_bottom)
    union = l1 * w1 + l2 * w2 - intersection
    return intersection / (union + 1e-8)


def nms_3d(boxes, scores, iou_threshold=0.5):
    if len(boxes) == 0:
        return []

    order = np.argsort(scores)[::-1]
    keep = []

    while len(order) > 0:
        i = order[0]
        keep.append(i)

        if len(order) == 1:
            break

        ious = np.array([compute_iou_3d(boxes[i], boxes[j]) for j in order[1:]])
        mask = ious < iou_threshold
        order = order[1:][mask]

    return keep


def post_process_predictions(cls_pred, reg_pred, dir_pred, conf_threshold=0.3, nms_threshold=0.5):
    num_classes, H, W = cls_pred.shape
    cls_scores = torch.sigmoid(cls_pred)
    max_scores, max_classes = cls_scores.max(dim=0)
    mask = max_scores > conf_threshold

    if mask.sum() == 0:
        return np.zeros((0, 7)), np.zeros((0,)), np.zeros((0,), dtype=np.int64)

    scores = max_scores[mask].cpu().numpy()
    classes = max_classes[mask].cpu().numpy()
    y_coords, x_coords = torch.where(mask)
    boxes_params = reg_pred[:, y_coords, x_coords].t().cpu().numpy()

    grid_size = 32
    scale = 100.0 / grid_size
    x_world = (x_coords.cpu().numpy() - grid_size / 2) * scale
    y_world = (y_coords.cpu().numpy() - grid_size / 2) * scale

    boxes = boxes_params.copy()
    boxes[:, 0] += x_world
    boxes[:, 1] += y_world

    final_boxes = []
    final_scores = []
    final_classes = []

    for cls_id in range(num_classes):
        cls_mask = classes == cls_id
        if cls_mask.sum() == 0:
            continue

        cls_boxes = boxes[cls_mask]
        cls_scores = scores[cls_mask]
        keep = nms_3d(cls_boxes, cls_scores, nms_threshold)

        if len(keep) > 0:
            final_boxes.append(cls_boxes[keep])
            final_scores.append(cls_scores[keep])
            final_classes.append(np.full(len(keep), cls_id))

    if len(final_boxes) == 0:
        return np.zeros((0, 7)), np.zeros((0,)), np.zeros((0,), dtype=np.int64)

    return np.vstack(final_boxes), np.hstack(final_scores), np.hstack(final_classes)

print("Post-processing ")

Post-processing 


In [16]:
# Visualizzazione
def visualize_detections(image, points, pred_boxes, pred_scores, pred_classes):
    fig = plt.figure(figsize=(20, 10))

    ax1 = plt.subplot(1, 2, 1)
    ax1.imshow(image)
    ax1.set_title('Camera Image')
    ax1.axis('off')

    ax2 = plt.subplot(1, 2, 2)
    mask = (np.abs(points[:, 0]) < 50) & (np.abs(points[:, 1]) < 50)
    filtered = points[mask]
    ax2.scatter(filtered[:, 0], filtered[:, 1], c=filtered[:, 2], cmap='viridis', s=0.5, alpha=0.5)

    colors = ['red', 'blue', 'green']
    names = ['Car', 'Pedestrian', 'Cyclist']

    for box, score, cls in zip(pred_boxes, pred_scores, pred_classes):
        x, y, z, l, w, h, yaw = box
        rect = Rectangle((x - l/2, y - w/2), l, w, fill=False,
                         color=colors[cls], linewidth=2, label=f'{names[cls]}: {score:.2f}')
        ax2.add_patch(rect)

    ax2.set_xlim(-50, 50)
    ax2.set_ylim(-50, 50)
    ax2.set_xlabel('X (meters)')
    ax2.set_ylabel('Y (meters)')
    ax2.set_title(f"Bird's Eye View - {len(pred_boxes)} detections")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print("Visualizzazione definita")

Visualizzazione definita


## TEST INFERENZA SU CAMPIONE

In [17]:
# Test su un campione
test_idx = 270




points_data, image_data, _, _ = dataset[test_idx]
points = points_data.unsqueeze(0).to(device)
image = image_data.unsqueeze(0).to(device)

print(f"🔍 Testing on frame {test_idx}")

model.eval()
with torch.no_grad():
    cls_pred, reg_pred, dir_pred = model(points, image)
    cls_pred = cls_pred[0]
    reg_pred = reg_pred[0]
    dir_pred = dir_pred[0]

    pred_boxes, pred_scores, pred_classes = post_process_predictions(
        cls_pred, reg_pred, dir_pred, conf_threshold=0.2
    )

    print(f"Detected: {len(pred_boxes)} objects")
    if len(pred_boxes) > 0:
        print(f"   Classes: {pred_classes}")
        print(f"   Scores: {pred_scores}")

image_np = image_data.permute(1, 2, 0).cpu().numpy()
points_np = points_data.cpu().numpy()

visualize_detections(image_np, points_np, pred_boxes, pred_scores, pred_classes)

NameError: name 'dataset' is not defined




-  Modello addestrato
-  Inferenza funzionante
-  Visualizzazione 3D bounding boxes
-  Post-processing con NMS

